# Holter 전처리 v2: Swee 2026 기준 + 3유도 PCA 합치기

**목적**: Swee 외 2026 전처리를 따라 30초 조각마다 특징 28개를 만든다. 단, **Y 유도 하나 대신 3유도(X·Y·Z)를 PCA로 섞어 만든 신호 1개**를 쓴다 (2026-10-27 결정). 특징 기반 모델(MIL-TCN, 전통 ML)의 입력으로 쓴다.

- 대상은 최종 사용 환자 **445명** (`final_holter_ecg_1_cohort.ipynb` ⑥ → `cohort_final.csv`, 2026-10-27 결정)
- 결과는 구글 드라이브 팀 폴더에 그대로 올릴 수 있게 저장한다: 특징 CSV → `ECG_num`, PCA로 합친 정제 신호 `.npy` → `ECG_tem_clean`. 파일 이름은 환자 ID(`P0001`), 증강본은 `P0001-1`(앞 15시간)·`P0001-2`(뒤 15시간). 원본 445개 + 증강본 890개 = 1,335개씩
- 유도 규칙이 Swee와 다르므로 **Swee 결과(ECG 단독 SCD AUROC 0.628)를 그대로 재현하는 노트북이 아니다.** 비교할 때 이 차이를 함께 적는다.
- 3유도를 섞는 방법은 같은 MUSIC 데이터를 쓴 Ramírez 2017의 lead-wise PCA에서 가져왔다. Ramírez는 T파 지표에 썼고, 우리는 R파·HRV 특징에 쓴다 (아이디어만 빌려 옴).
- 기존 v1(`holter_preprocessing*.ipynb`, 3유도 원파형 DL용)과는 **별개**다. v1 결과는 건드리지 않는다.
- 잡음을 완전히 잡지 않아도 된다. Swee는 잡음 조각을 빼지 않고, **품질 점수를 특징으로 넣는** 방식이다.
- Swee 규칙의 출처는 논문 본문이 아니라 **공개 코드**다. 본문에 없는 규칙(앞 30초 버림, R파 3개 미만 조각 버림 등)도 코드에 있으면 따른다.

| 출처 | 링크 |
|---|---|
| 논문 (medRxiv, 동료 심사 전) | Swee S, et al. *Multimodal Integration of Ambulatory ECG and Clinical Features for Sudden Cardiac Death and Pump Failure Death Prediction.* medRxiv 2026. [본문](https://www.medrxiv.org/content/10.64898/2026.04.21.26351421v2.full) |
| 코드 (GPL-3.0) | [github.com/stswee/multimodal_aecg_clinicalfeatures](https://github.com/stswee/multimodal_aecg_clinicalfeatures). 코드를 복사하지 않고 규칙만 보고 다시 작성함 |
| 3유도 합치기 | Ramírez J, et al. *T-Wave Morphology Restitution Predicts Sudden Cardiac Death in Patients With Chronic Heart Failure.* J Am Heart Assoc 2017;6:e005310. [PMC5524085](https://pmc.ncbi.nlm.nih.gov/articles/PMC5524085) (MUSIC 651명). 원문: "Lead-wise principal component analysis was applied to choose the ECG projection that better emphasized the T-wave energy." |

## 전처리 기준

| 순서 | 규칙 | 값 | 출처 |
|---|---|---|---|
| 1 | 유도 | **있는 유도 모두** 읽음 (X·Y·Z, 2유도 환자는 X·Y) | Ramírez 2017 (3유도 사용). Swee는 Y 하나 (`p_signal[:, 1]`) |
| 2 | 기록 앞부분 | **30초 버림** | Swee `0_Preprocessing/preprocess_single_ecg_HRV_complete.py`, `--skip_seconds` 기본값 |
| 3 | 대역통과 필터 | 4차 Butterworth 0.5–40 Hz, `filtfilt` (앞뒤로 걸어 위치가 밀리지 않음). **유도마다 따로** | Swee 같은 파일, `bandpass_filter` (Swee는 Y에만) |
| 4 | 기저선 제거 | 0.8초(161 샘플) median filter로 구한 기저선을 뺌. **유도마다 따로** | Swee 같은 파일, `baseline_correction` (Swee는 Y에만) |
| 5 | **3유도 합치기** | 필터 후 유도들로 PCA → **첫 번째 주성분 신호 1개**. 환자마다 **기록 전체에서 한 번** 계산. 부호는 Y와 같은 방향 | 방법: Ramírez 2017 (lead-wise PCA). 범위·주성분 선택·부호: ✋ 자체 판단 (아래 설명) |
| 6 | R파 검출 | NeuroKit2 `ecg_peaks` 기본 방법, **합친 신호의 기록 전체에서 한 번** | Swee 같은 파일 |
| 7 | 조각 | 처음부터 겹치지 않게 30초씩, 끝에 남는 30초 미만은 버림 | Swee `1_Segmenting/create_window_index_metadata.py` |
| 8 | 조각 제외 | **R파 3개 미만이면 버림** (Swee의 유일한 조각 제외 규칙) | Swee `1_Segmenting/segment_all_patients_by_start_indices_HRV_complete.py` |
| 9 | 특징 | 아래 28개. R파 5개 미만 조각은 NeuroKit 특징(심박수·품질·HRV 25개)이 빈 값 | Swee 같은 파일 + `3_ECG_Modeling/train_tcn_mil_hrv_csv_multiclass_SCDPFD_complete.py` (`ALLOWED_FEATURES`) |
| 10 | 환자 제외 | 남은 조각 3개 미만 → 모델에서 제외 (여기서는 **표시만**) | Swee `3_ECG_Modeling/...` (`--min_segments`) |
| 11 | 정제 신호 저장 | 5번의 합친 신호를 **float16 `.npy`**로 저장 (`P0001.npy`). 기록 시작 30초 뒤부터 | 사용자 결정 2026-10-27 (구글 드라이브 `ECG_tem_clean`) |
| 12 | 증강본 | 정제된 합친 신호의 **앞 15시간(`-1`)·뒤 15시간(`-2`)**을 잘라 `.npy`와 특징 CSV를 원본과 같은 방식으로 저장. R파는 원본 전체에서 찾은 위치를 그대로 씀 | 증강 규칙: `final_holter_ecg_2_augmentation.ipynb` (15시간, 맨 앞·맨 뒤). 정제한 뒤에 자르는 것은 ✋ 자체 판단 (아래 설명) |

**12번 증강본을 정제한 뒤에 자르는 이유**

- 원본과 증강본이 같은 필터, 같은 PCA 가중치를 거친다. 증강본마다 따로 정제하면 PCA 가중치가 조금씩 달라진다.
- 노트북 2의 WFDB 증강 파일은 원파형 그대로이고, `-1`에는 앞 30초 보정 신호가 들어 있다. 이 노트북은 그 파일을 읽지 않는다. 그래서 여기서 만드는 `-1`은 보정 신호를 뺀 실제 심전도 15시간이고, 노트북 2 파일보다 30초 늦게 시작한다.
- 30초 조각의 경계는 증강본의 시작점부터 다시 센다. 그래서 같은 시각이라도 원본 조각과 경계가 다를 수 있다.

**5번 3유도 합치기 자세히**

- PCA는 세 유도의 값을 시각마다 가중치로 더한 새 신호를 만든다: `합친 신호 = w_X·X + w_Y·Y + w_Z·Z`. 가중치는 세 유도가 함께 가장 크게 움직이는 방향이고, 환자마다 데이터에서 계산한다.
- **첫 번째 주성분**을 고른 이유: 분산이 가장 큰 방향이고, 심전도에서는 진폭이 가장 큰 QRS가 분산을 주로 만든다. R파 검출에 맞는 신호다. Ramírez는 T파 에너지가 가장 큰 투영을 골랐다(어느 성분인지는 원문에 없음).
- **부호**: 주성분의 부호(+/−)는 수학적으로 정해지지 않는다. Y 가중치가 0 이상이 되게 맞춰서, 합친 신호가 Y와 같은 방향을 향하게 한다 (Swee가 쓰던 신호와 극성이 가장 비슷하게).
- 환자마다 가중치(`pca_w_X/Y/Z`)와 첫 번째 주성분이 설명하는 분산 비율(`pca_explained`)을 처리 기록에 남긴다.

**특징 28개**

| 묶음 | 특징 | 계산 |
|---|---|---|
| HRV 시간 영역 (13) | MeanNN, MedianNN, SDNN, RMSSD, SDSD, CVNN, CVSD, IQRNN, MadNN, pNN20, pNN50, MinNN, MaxNN | `nk.hrv_time` |
| HRV 주파수 (3) | HF, LnHF, TP | `nk.hrv_frequency` |
| HRV 비선형 (5) | SD1, SD2, SD1SD2, S, SampEn | `nk.hrv_nonlinear` |
| R파 간격 (3) | n_rpeaks, pvc_burden_pct (RR < 0.8 × 중앙값 RR 비율), longest_rr_pause | 직접 계산 |
| 심박수 (2) | hr_mean, hr_std | 조각에 `nk.ecg_process` → `ECG_Rate` |
| 품질 (2) | ecg_quality_mean, ecg_quality_low_frac (품질 < 0.8 비율) | 조각에 `nk.ecg_process` → `ECG_Quality` |

HRV 21개는 Swee처럼 세 함수를 한 번에 계산한다. **하나라도 실패하면 21개 모두 빈 값**이다. 모든 특징은 합친 신호 1개에서 계산한다.

## Swee 코드와 다른 점

| 다른 점 | 이유 | 결과에 영향 |
|---|---|---|
| **Y 하나 대신 3유도를 PCA로 합친 신호 1개 사용** (필터·기저선 제거는 유도마다 한 뒤 합침) | 사용자 결정 (2026-10-27). 근거: Ramírez 2017 | **있음** (R파 위치, 품질 점수, 모든 특징) |
| 기록 전체 `nk.ecg_process` 생략 | Swee는 저장만 하고 28개 특징 계산에는 쓰지 않음. 24시간 전체라 메모리를 많이 씀 | 없음 |
| 28개 밖의 특징(`ecg_delineate`의 QRS·QT, 신호 통계 등) 생략 | 모델에 안 들어감. 별도 `try` 블록이라 28개 값에 영향 없음 | 없음 |
| 조각의 `nk.ecg_process`가 오류를 내면: Swee는 **프로그램이 멈춤**, 우리는 그 조각의 NeuroKit 특징을 빈 값으로 두고 개수(`n_nk_fail`)를 기록 | 멈추면 전체 실행이 불가능 | 있음 (⑥에서 개수 확인) |
| 환자 ID를 `P0001` 형식으로 저장 (Swee는 `0001`) | 우리 다른 표와 맞춤 | 없음 |
| 여러 환자를 동시에 처리 (`joblib`) | 시간 단축 | 없음 |

## 결정 기록 (사용자가 정한 것만)

| 날짜 | 결정 | 근거 |
|---|---|---|
| 2026-10-26 | v2는 Swee 2026 코드의 전처리 규칙만 사용, 우리 기준(1~3단계)은 더하지 않음 (10-27에 유도 규칙만 바꿈) | Swee 2026 (MUSIC, 같은 데이터) |
| 2026-10-26 | 잡음을 완전히 잡지 않아도 됨 | 특징 기반 모델이고, 품질 점수가 특징으로 들어감 |
| 2026-10-26 | 새 노트북으로 시작 (v1과 분리) | — |
| 2026-10-27 | **Y 유도 하나만 쓰는 방식은 버리고, 3유도를 섞어 만든 신호 1개만 사용** (이 방향으로 감) | Ramírez 2017 JAHA (MUSIC, lead-wise PCA) |
| 2026-10-27 | 전처리 대상은 최종 사용 환자 445명 (팀 분할 파일에서 등록 전 기기 이식 63명 제외, 정답은 4년 SCD) | Swee 2026 코호트 규칙 (`2_Labeling`) |
| 2026-10-27 | 정제 신호를 **PCA로 합친 신호 1개, float16 `.npy`**로 저장. 파일 이름은 환자 ID, 증강본은 `-1`·`-2`. 증강본의 `.npy`와 특징 CSV도 이 노트북에서 만듦 | 구글 드라이브 팀 폴더 (`ECG_num`, `ECG_tem_clean`) |
| 2026-10-27 | **neurokit2 0.2.13** 사용 (pandas 2.3). 0.2.12는 numpy 2.5에서 없어진 `np.trapz`를 써서 HRV 주파수·비선형 계산이 모든 조각에서 실패함 (시험 환자 HRV 21개 100% 빈 값). 0.2.13은 이 문제와 pandas 3의 빈 값 처리 문제를 고친 버전 | 시험 결과 (④) |
| 2026-10-27 | 원본 Holter를 다시 받아 **학교 컴퓨터에서 전처리 실행** (C: 여유 약 145 GB, 결과 약 35 GB) | 사용자 요청 |
| 2026-10-27 | 교육용 노트북은 용량이 부족해 원본·전처리·증강 파일을 저장할 수 없음 → 전처리와 증강은 집 컴퓨터에서 하고, 결과를 구글 드라이브에 올려서 사용 | 교육용 노트북 저장 용량 부족 |

## 검토 중 (제안, 아직 결정 안 함)

| 항목 | 내용 |
|---|---|
| PCA 범위 | 지금은 환자마다 기록 전체(약 24시간)에서 한 번. Ramírez 원문에는 범위가 없음. 시간대마다(예: 조각마다) 다시 계산하는 방법도 있음 |
| 주성분 선택 | 지금은 첫 번째 주성분 (분산 최대 = 주로 QRS). Ramírez는 T파 에너지 기준으로 골랐고 어느 성분인지는 적지 않음 |
| 잡음 큰 유도 | 한 유도에 큰 움직임 잡음이 오래 있으면 그 유도가 분산을 차지해서 주성분 가중치가 그쪽으로 쏠릴 수 있음. ⑥에서 가중치가 가장 큰 유도와 `pca_explained`를 보고 판단 |
| R파 극성 | 부호를 Y에 맞춰도, 합친 신호의 R파가 아래로 향하는 환자가 있을 수 있음 (Swee의 Y도 같은 문제). ④-2 그림으로 확인 |
| Swee 기준선과 비교 | Y 방식은 쓰지 않기로 해서, PCA가 Y보다 나은지는 직접 비교하지 않음. Swee 공개 결과(0.628)와는 조건이 달라 참고로만 비교 |
| NeuroKit2 버전 | 우리 0.2.13, Swee 버전은 저장소에 없음. 버전 차이로 R파·품질 점수가 조금 다를 수 있음 |
| `nk_fail` 조각 | ⑥에서 개수를 보고, 많으면 처리 방법을 다시 정함 |

## 진행 상황

| 셀 | 내용 | 상태 | 날짜 |
|---|---|---|---|
| ① | 설정 | | |
| ② | 기록 1개 전처리 함수 (3유도 PCA 포함) | | |
| ③ | 30초 조각 특징 함수 | | |
| ④ | 환자 1명 처리 + 시험 | | |
| ④-2 | 3유도와 합친 신호 그림 | | |
| ⑤ | 전체 실행 (445명) | | |
| ⑥ | 결과 확인 | | |

In [ ]:
# ① 설정: 경로, 전처리 상수 (Swee 2026 + 3유도 PCA), 특징 28개 목록
from pathlib import Path
import resource
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.signal as sp
import wfdb
import neurokit2 as nk
from joblib import Parallel, delayed
from tqdm.auto import tqdm

plt.rcParams["font.family"] = "NanumGothic"       # 그림의 한글 (다른 전처리 노트북과 같은 글꼴)
plt.rcParams["axes.unicode_minus"] = False

# 경로
RAW_DIR = Path("/mnt/c/data/music/Holter_ECG")            # PhysioNet 원본 (.hea / .dat)
OUT_DIR = Path("/mnt/c/data/music/SCD_Holter_v2_pca")     # 결과 데이터 (원본에서 나온 값이라 저장소 밖에 둠)
FEATURE_DIR = OUT_DIR / "segment_features"                # 환자마다 CSV 1개: 조각 1줄 = 특징 28개 (드라이브 ECG_num)
CLEAN_DIR = OUT_DIR / "clean_signal"                      # 환자마다 .npy 1개: PCA로 합친 정제 신호 float16 (드라이브 ECG_tem_clean)
LOG_PATH = Path("../results/preprocess_swee/record_log.csv")   # 환자마다 1줄: 처리 상태, 유도, PCA 가중치, 조각 수, 시간
COHORT_PATH = Path("../results/final_holter_ecg/cohort_final.csv")   # 처리할 환자 = 최종 사용 환자 445명 (노트북 1 ⑥)

# 전처리 상수 (위 '전처리 기준' 표의 순서 번호)
ALL_LEADS = ["X", "Y", "Z"]   # 1. MUSIC Holter 유도 이름. 2유도 환자는 Z가 없음 → 기록에서 Z 가중치는 빈 값
SKIP_SEC = 30             # 2. 기록 앞 30초 버림
BAND_HZ = (0.5, 40)       # 3. 대역통과 필터 범위
FILTER_ORDER = 4          # 3. 필터 차수
BASELINE_SEC = 0.8        # 4. 기저선 median filter 길이
SIGN_LEAD = "Y"           # 5. 주성분 부호를 이 유도와 같은 방향으로 맞춤
WINDOW_SEC = 30           # 7. 조각 길이
MIN_RPEAKS_SEGMENT = 3    # 8. R파가 이보다 적은 조각은 버림
MIN_RPEAKS_NEUROKIT = 5   # 9. R파가 이보다 적으면 NeuroKit 특징은 빈 값
PVC_RR_RATIO = 0.8        # 9. RR이 중앙값 RR의 0.8배보다 짧으면 조기박동으로 셈
QUALITY_LOW = 0.8         # 9. NeuroKit 품질 점수가 이보다 낮으면 '낮은 품질'
MIN_SEGMENTS = 3          # 10. 조각이 이보다 적은 환자는 모델에서 제외 (여기서는 표시만)
AUG_HOURS = 15            # 12. 증강본 길이: 정제된 신호의 앞 15시간(-1), 뒤 15시간(-2). 노트북 2와 같은 값

# 모델 입력 특징 28개 (Swee ALLOWED_FEATURES와 같은 순서)
FEATURES = [
    # HRV 시간 영역 (nk.hrv_time)
    "HRV_MeanNN", "HRV_MedianNN", "HRV_SDNN", "HRV_RMSSD", "HRV_SDSD", "HRV_CVNN", "HRV_CVSD",
    "HRV_IQRNN", "HRV_MadNN", "HRV_pNN20", "HRV_pNN50", "HRV_MinNN", "HRV_MaxNN",
    # HRV 비선형 (nk.hrv_nonlinear)
    "HRV_SD1", "HRV_SD2", "HRV_SD1SD2", "HRV_S",
    # HRV 주파수 (nk.hrv_frequency)
    "HRV_HF", "HRV_LnHF", "HRV_TP",
    # HRV 비선형 (nk.hrv_nonlinear)
    "HRV_SampEn",
    # R파 간격 (직접 계산)
    "n_rpeaks", "pvc_burden_pct", "longest_rr_pause",
    # 심박수 · 품질 (조각에 nk.ecg_process)
    "hr_mean", "hr_std", "ecg_quality_mean", "ecg_quality_low_frac",
]
ID_COLUMNS = ["patient_id", "window_idx", "start_idx"]
PCA_COLUMNS = [f"pca_w_{lead}" for lead in ALL_LEADS] + ["pca_explained"]
LOG_COLUMNS = ["patient_id", "status", "error", "leads", "n_leads", "fs", "n_samples_raw", "n_nan_raw",
               *PCA_COLUMNS, "n_rpeaks", "n_windows", "n_segments_kept", "n_nk_fail", "enough_segments",
               "n_aug", "n_segments_aug1", "n_segments_aug2", "n_nk_fail_aug",
               "seconds", "peak_mem_gb"]

N_JOBS = 8                # 동시에 처리할 환자 수. ④ 시험 peak_mem_gb 1.24 GB × 8 ≈ 10 GB → RAM 16 GB 안 (3유도라 Y만 읽을 때보다 메모리 큼)

FEATURE_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", message="DFA_alpha2 related indices will not be calculated")   # Swee와 같은 경고 숨김
warnings.filterwarnings("ignore", message="invalid value encountered in scalar divide")

# 처리할 기록 목록: RECORDS 파일의 Holter 기록(936개) 중 최종 사용 환자만
records_file = RAW_DIR.parent / "RECORDS"
holter_ids = [line.split("/")[-1] for line in records_file.read_text().split() if line.startswith("Holter_ECG/")]
cohort_ids = set(pd.read_csv(COHORT_PATH)["pid"])
record_ids = [r for r in holter_ids if r in cohort_ids]
assert len(record_ids) == len(cohort_ids), f"Holter 기록이 없는 환자: {sorted(cohort_ids - set(holter_ids))}"
print(f"Holter 기록 {len(holter_ids)}개 중 처리 대상 {len(record_ids)}명 · neurokit2 {nk.__version__} · wfdb {wfdb.__version__}")

In [ ]:
# ② 기록 1개 전처리: 읽기 → 앞 30초 버림 → 유도마다 0.5–40 Hz 필터·기저선 제거 → 3유도 PCA로 신호 1개 → R파 검출
def read_leads(record_id):
    """원본의 모든 유도를 mV로 읽음 → ((샘플 수, 유도 수) 신호, 샘플링 속도, 유도 이름 목록)"""
    record = wfdb.rdrecord(str(RAW_DIR / record_id))
    return record.p_signal, record.fs, record.sig_name


def bandpass_filter(signal, fs):
    """4차 Butterworth 0.5–40 Hz를 앞뒤로 한 번씩 걸음 (filtfilt: 파형 위치가 밀리지 않음)"""
    nyquist = fs / 2
    b, a = sp.butter(FILTER_ORDER, [BAND_HZ[0] / nyquist, BAND_HZ[1] / nyquist], btype="band")
    return sp.filtfilt(b, a, signal)


def remove_baseline(signal, fs):
    """0.8초 median filter로 느린 기저선을 구해서 뺌 (창 길이는 홀수여야 해서 161 샘플)"""
    kernel = int(BASELINE_SEC * fs // 2 * 2 + 1)
    return signal - sp.medfilt(signal, kernel_size=kernel)


def filter_leads(raw, fs):
    """(샘플 수, 유도 수) 원본 → 앞 30초를 버리고 유도마다 필터·기저선 제거 (Swee가 Y에 하던 처리를 유도마다 그대로)"""
    trimmed = raw[int(SKIP_SEC * fs):]
    leads = np.empty(trimmed.shape, dtype=np.float32)
    for j in range(trimmed.shape[1]):
        leads[:, j] = remove_baseline(bandpass_filter(trimmed[:, j], fs), fs)
    return leads


def combine_leads_pca(leads, lead_names):
    """필터 후 유도들 → (첫 번째 주성분 신호, 유도별 가중치 dict, 첫 번째 주성분이 설명하는 분산 비율)
    부호는 수학적으로 정해지지 않아서 SIGN_LEAD(Y) 가중치가 0 이상이 되게 맞춤"""
    centered = leads - leads.mean(axis=0)
    eigvals, eigvecs = np.linalg.eigh(np.cov(centered, rowvar=False))   # 고윳값 오름차순 → 마지막 열이 첫 번째 주성분
    weights = eigvecs[:, -1]
    if weights[lead_names.index(SIGN_LEAD)] < 0:
        weights = -weights
    combined = (centered @ weights).astype(np.float32)      # 합친 신호 = w_X·X + w_Y·Y + w_Z·Z
    return combined, dict(zip(lead_names, weights)), eigvals[-1] / eigvals.sum()


def preprocess_record(record_id):
    """기록 1개 → (합친 신호 float32, R파 위치 배열, 기록 정보 dict)"""
    raw, fs, lead_names = read_leads(record_id)
    info = {"leads": "".join(lead_names), "n_leads": len(lead_names), "fs": fs, "n_samples_raw": len(raw),
            "n_nan_raw": int(np.isnan(raw).sum())}           # 모든 유도의 NaN 합. Swee는 NaN을 처리하지 않음 → 있으면 여기서 보임

    leads = filter_leads(raw, fs)
    del raw
    clean, weights, explained = combine_leads_pca(leads, lead_names)
    info.update({f"pca_w_{lead}": weights.get(lead, np.nan) for lead in ALL_LEADS})
    info["pca_explained"] = explained

    _, peaks = nk.ecg_peaks(clean, sampling_rate=fs)        # 합친 신호의 기록 전체에서 R파를 한 번만 찾음
    rpeaks = peaks["ECG_R_Peaks"].astype(np.int32)
    info["n_rpeaks"] = len(rpeaks)
    return clean, rpeaks, info

In [ ]:
# ③ 30초 조각 특징: 조각 자르기 → R파 3개 미만이면 버림 → 특징 28개
def rr_features(rpeaks_seg, fs):
    """R파 간격으로 계산하는 특징 3개: R파 수, 조기박동 비율(%), 가장 긴 RR(ms)"""
    rr_ms = np.diff(rpeaks_seg) / fs * 1000
    early = rr_ms < PVC_RR_RATIO * np.median(rr_ms)         # 평소보다 많이 빨리 온 박동 = 조기박동으로 셈 (Swee 정의)
    return {"n_rpeaks": len(rpeaks_seg),
            "pvc_burden_pct": 100 * early.sum() / len(rr_ms),
            "longest_rr_pause": rr_ms.max()}


def neurokit_features(segment, rpeaks_seg, fs):
    """NeuroKit2로 계산하는 특징 25개: 심박수 2 + 품질 2 + HRV 21. R파가 5개 미만이면 빈 dict"""
    if len(rpeaks_seg) < MIN_RPEAKS_NEUROKIT:
        return {}

    signals, _ = nk.ecg_process(segment, sampling_rate=fs)   # 조각 안에서 다시 처리 (Swee와 같음)
    hr = signals["ECG_Rate"].to_numpy()
    quality = signals["ECG_Quality"].to_numpy()
    feats = {"hr_mean": hr.mean(), "hr_std": hr.std(),
             "ecg_quality_mean": quality.mean(),
             "ecg_quality_low_frac": np.mean(quality < QUALITY_LOW)}

    try:   # Swee처럼 세 함수를 한 번에: 하나라도 실패하면 HRV 21개 모두 빈 값
        hrv = pd.concat([nk.hrv_time(rpeaks_seg, sampling_rate=fs),
                         nk.hrv_frequency(rpeaks_seg, sampling_rate=fs),
                         nk.hrv_nonlinear(rpeaks_seg, sampling_rate=fs)], axis=1)
        feats.update(hrv.iloc[0].to_dict())
    except Exception:
        pass
    return feats


def segment_features(clean, rpeaks, fs, patient_id):
    """기록 전체 → (조각별 특징 표, 전체 조각 수, NeuroKit 오류 조각 수)"""
    window = WINDOW_SEC * fs
    n_windows = len(clean) // window                        # 끝에 남는 30초 미만은 버림
    rows, n_nk_fail = [], 0

    for i in range(n_windows):
        start = i * window
        in_window = (rpeaks >= start) & (rpeaks < start + window)
        rpeaks_seg = rpeaks[in_window] - start              # 조각 안 위치로 바꿈
        if len(rpeaks_seg) < MIN_RPEAKS_SEGMENT:
            continue                                        # R파 3개 미만 → 조각 버림

        feats = rr_features(rpeaks_seg, fs)
        try:
            feats.update(neurokit_features(clean[start:start + window], rpeaks_seg, fs))
        except Exception:
            n_nk_fail += 1                                  # Swee는 여기서 멈춤 → 우리는 빈 값으로 두고 셈
        rows.append({"patient_id": patient_id, "window_idx": i, "start_idx": start, **feats})

    table = pd.DataFrame(rows).reindex(columns=ID_COLUMNS + FEATURES)   # 28개만 남김, 없는 값은 NaN
    return table, n_windows, n_nk_fail

In [ ]:
# ④ 환자 1명 처리 함수 + 시험: 전처리 → 조각 특징 → CSV·.npy 저장 → 증강본 2개도 같은 방식 → 처리 기록 1줄
def save_clean(signal, name):
    """합친 신호를 float16 .npy로 저장. 파일 이름 = 환자 ID 또는 증강 ID"""
    np.save(CLEAN_DIR / f"{name}.npy", signal.astype(np.float16))


def augment_crops(n_samples, fs):
    """정제된 신호 길이 → [(번호, 시작, 끝)]: 앞 15시간(-1), 뒤 15시간(-2). 15시간 이하면 빈 목록"""
    crop_len = int(AUG_HOURS * 3600 * fs)
    if n_samples <= crop_len:
        return []
    return [(1, 0, crop_len), (2, n_samples - crop_len, n_samples)]


def process_patient(record_id):
    """환자 1명을 끝까지 처리하고 기록 1줄(dict)을 돌려줌. 오류가 나도 멈추지 않고 이유를 남김"""
    t0 = time.time()
    log = {"patient_id": record_id, "status": "ok", "error": ""}
    try:
        clean, rpeaks, info = preprocess_record(record_id)
        log.update(info)
        fs = info["fs"]
        table, n_windows, n_nk_fail = segment_features(clean, rpeaks, fs, record_id)
        table.to_csv(FEATURE_DIR / f"{record_id}.csv", index=False)
        save_clean(clean, record_id)
        log.update({"n_windows": n_windows, "n_segments_kept": len(table), "n_nk_fail": n_nk_fail,
                    "enough_segments": len(table) >= MIN_SEGMENTS})

        crops = augment_crops(len(clean), fs)
        log.update({"n_aug": len(crops), "n_nk_fail_aug": 0})
        for k, start, end in crops:   # 증강본: 정제된 원본에서 잘라 원본과 같은 방식으로 저장
            aug_id = f"{record_id}-{k}"
            in_crop = (rpeaks >= start) & (rpeaks < end)
            aug_table, _, aug_fail = segment_features(clean[start:end], rpeaks[in_crop] - start, fs, aug_id)
            aug_table.to_csv(FEATURE_DIR / f"{aug_id}.csv", index=False)
            save_clean(clean[start:end], aug_id)
            log[f"n_segments_aug{k}"] = len(aug_table)
            log["n_nk_fail_aug"] += aug_fail
    except Exception as e:
        log.update({"status": "error", "error": f"{type(e).__name__}: {e}"})
    log["seconds"] = round(time.time() - t0, 1)
    log["peak_mem_gb"] = round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, 2)   # 이 프로세스의 최대 메모리
    return log


TEST_ID = "P0307"   # v1에서 확인한 깨끗한 정상 동리듬 환자

test_log = process_patient(TEST_ID)
display(pd.Series(test_log).to_frame("값"))

test_table = pd.read_csv(FEATURE_DIR / f"{TEST_ID}.csv")
display(test_table.head())
display(test_table[FEATURES].isna().mean().round(3).to_frame("빈 값 비율").T)

for path in sorted(CLEAN_DIR.glob(f"{TEST_ID}*.npy")):   # 원본 1개 + 증강본 2개
    signal = np.load(path)
    print(f"{path.name}: {signal.shape} {signal.dtype} · {len(signal) / test_log['fs'] / 3600:.2f}시간 · {path.stat().st_size / 1e6:.0f} MB")

In [ ]:
# ④-2 그림: 시험 환자의 필터 후 유도들과 합친 신호를 같은 10초로 겹쳐 봄 (부호·R파 방향 확인)
#     전처리를 한 번 더 돌리므로 ④보다 조금 걸림
PLOT_START_H = 1.0        # 그릴 위치 (앞 30초를 버린 뒤 시각, 시간)
PLOT_SECONDS = 10

raw, fs, lead_names = read_leads(TEST_ID)
leads = filter_leads(raw, fs)
del raw
combined, weights, explained = combine_leads_pca(leads, lead_names)

a = int(PLOT_START_H * 3600 * fs)
b = a + PLOT_SECONDS * fs
t = np.arange(b - a) / fs
fig, axes = plt.subplots(len(lead_names) + 1, 1, figsize=(12, 1.8 * (len(lead_names) + 1)), sharex=True)
for ax, j in zip(axes, range(len(lead_names))):
    ax.plot(t, leads[a:b, j], lw=0.8, color="gray")
    ax.set_ylabel(f"{lead_names[j]} (mV)")
    ax.set_title(f"{lead_names[j]} · 가중치 {weights[lead_names[j]]:+.2f}", fontsize=9, loc="left")
axes[-1].plot(t, combined[a:b], lw=0.8, color="tab:blue")
axes[-1].set_ylabel("합친 신호")
axes[-1].set_title(f"첫 번째 주성분 · 설명 분산 {explained:.1%}", fontsize=9, loc="left")
axes[-1].set_xlabel("초")
fig.suptitle(f"{TEST_ID}: 필터 후 유도와 PCA로 합친 신호 ({PLOT_START_H}h부터 {PLOT_SECONDS}초)", fontsize=10)
fig.tight_layout()
plt.show()
del leads, combined

📝 **④ 결과 메모** (실행 후 적기)
- 처리 시간 (`seconds`) → 445명 예상 시간 = 시간 × 445 ÷ `N_JOBS`:
- 최대 메모리 (`peak_mem_gb`) → `N_JOBS` 몇으로 할지:
- 유도 구성 (`leads`), PCA 가중치 (`pca_w_X/Y/Z`), 설명 분산 (`pca_explained`):
- ④-2 그림: 합친 신호의 R파가 위로 향하는가, 잡음이 커졌거나 줄었는가:
- 버린 조각 비율 (1 − `n_segments_kept` / `n_windows`):
- 빈 값이 많은 특징:
- `.npy` 3개 (원본, `-1`, `-2`)의 길이·dtype·용량, 증강본 처리로 늘어난 시간:

In [ ]:
# ⑤ 전체 실행: 아직 기록이 없는 환자만, N_JOBS명씩 동시에 처리
#    환자 1명이 끝날 때마다 LOG_PATH에 바로 이어 씀 → 중간에 끊겨도 다시 실행하면 남은 환자부터 이어서 함
#    오류 난 환자도 기록에 남아서 다시 하지 않음. 다시 하려면 LOG_PATH에서 그 줄을 지우고 실행
done = set(pd.read_csv(LOG_PATH)["patient_id"]) if LOG_PATH.exists() else set()
todo = [r for r in record_ids if r not in done]
print(f"남은 환자 {len(todo)}명 / 전체 {len(record_ids)}명 · 동시 처리 {N_JOBS}명")

jobs = Parallel(n_jobs=N_JOBS, return_as="generator_unordered")(delayed(process_patient)(r) for r in todo)
for log in tqdm(jobs, total=len(todo), mininterval=30):
    row = pd.DataFrame([log]).reindex(columns=LOG_COLUMNS)
    row.to_csv(LOG_PATH, mode="a", header=not LOG_PATH.exists(), index=False)

In [ ]:
# ⑥ 결과 확인: 처리 상태, 유도 구성, PCA 가중치, 조각 수, 버린 조각 비율, NeuroKit 오류
log = pd.read_csv(LOG_PATH)
print("처리 상태:", log["status"].value_counts().to_dict())
print("유도 구성:", log["leads"].value_counts(dropna=False).to_dict())   # XYZ 또는 XY

errors = log[log["status"] == "error"]
if len(errors):
    display(errors[["patient_id", "error"]])

ok = log[log["status"] == "ok"].copy()
ok["drop_pct"] = 100 * (1 - ok["n_segments_kept"] / ok["n_windows"])   # R파 3개 미만으로 버린 조각 비율
summary_cols = ["n_windows", "n_segments_kept", "drop_pct", "n_nk_fail", "n_nan_raw", "seconds", "peak_mem_gb"]
display(ok[summary_cols].describe().round(1))

# PCA: 가중치가 가장 큰 유도 = 합친 신호를 주로 만든 유도. 한쪽으로 몰리거나 설명 분산이 낮은 환자는 그림으로 확인
display(ok[PCA_COLUMNS].describe().round(2))
print("가중치가 가장 큰 유도:", ok[PCA_COLUMNS[:-1]].abs().idxmax(axis=1).str[-1].value_counts().to_dict())
display(ok.nsmallest(5, "pca_explained")[["patient_id", "leads", *PCA_COLUMNS]].round(2))

print("NaN이 있는 기록:", (ok["n_nan_raw"] > 0).sum(), "명")
print("NeuroKit 오류 조각이 있는 환자:", (ok["n_nk_fail"] > 0).sum(), "명")
print(f"조각 {MIN_SEGMENTS}개 미만 환자 (모델에서 제외 대상):", (~ok["enough_segments"].astype(bool)).sum(), "명")

# 드라이브에 올릴 파일: 원본 + 증강본 이름이 같은 짝으로 다 있는지 (기대: 445 + 890 = 1,335개씩)
npy_names = {p.stem for p in CLEAN_DIR.glob("*.npy")}
csv_names = {p.stem for p in FEATURE_DIR.glob("*.csv")}
print("증강본 개수별 환자:", ok["n_aug"].value_counts().to_dict())
print(".npy", len(npy_names), "개 · 특징 CSV", len(csv_names), "개 · 짝이 안 맞는 이름:", sorted(npy_names ^ csv_names))
print(f".npy 총용량 {sum(p.stat().st_size for p in CLEAN_DIR.glob('*.npy')) / 1e9:.1f} GB")

📝 **⑥ 결과 메모** (실행 후 적기)
- 처리 상태 (성공 / 오류 수, 오류 이유):
- 유도 구성 (XYZ / XY 몇 명):
- PCA: 가중치가 가장 큰 유도 분포, 설명 분산 (중앙값, 최소), 설명 분산이 낮은 환자를 ④-2 방식으로 본 결과:
- 버린 조각 비율 (중앙값, 최대):
- NaN 기록, NeuroKit 오류 조각:
- 조각 3개 미만 환자:
- `.npy`·특징 CSV 개수 (기대 1,335개씩), 짝이 안 맞는 이름, `.npy` 총용량:
- 다음 단계: 조각 3개 미만 환자를 학습에서 뺄지 정하기 (Swee는 학습에서 뺌, `--min_segments`)